# FlashAttention forward and backward, step by step

This notebook follows [Part I: Exact Blocks, Rings, and Sparsity](https://g-u-n.github.io/blogs/scaling-long-context-attention.html). We implement FlashAttention forward, save the row-wise LSE, reconstruct probability tiles in backward, and compare the resulting gradients with PyTorch.

Every Triton kernel and launcher is defined directly in this notebook. Run the cells in order; the forward and backward sections each end with an independent correctness check.

## 0. Connect a GPU and prepare Triton

Colab should request a GPU runtime from the notebook metadata. If CUDA is unavailable, select **Runtime → Change runtime type → GPU**. We use fp16 by default because it works on T4 as well as newer Colab GPUs.

In [ ]:
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select Runtime → Change runtime type → GPU.")

try:
    import triton
    import triton.language as tl
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton
    import triton.language as tl

DEVICE = "cuda"
DTYPE = torch.float16
torch.manual_seed(0)

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("dtype used below:", DTYPE)
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
def check_close(name, got, reference, *, atol=3e-2, rtol=3e-2):
    torch.testing.assert_close(got, reference, atol=atol, rtol=rtol)
    max_error = (got.float() - reference.float()).abs().max().item()
    print(f"{name:<24} shape={tuple(got.shape)!s:<22} max error={max_error:.3e}")


def attention_reference(q, k, v, *, causal):
    scale = q.shape[-1] ** -0.5
    scores = torch.matmul(q.float(), k.float().transpose(-1, -2)) * scale
    if causal:
        mask = torch.ones(scores.shape[-2:], device=q.device, dtype=torch.bool).tril()
        scores = scores.masked_fill(~mask, -float("inf"))
    output = torch.matmul(torch.softmax(scores, dim=-1), v.float()).to(q.dtype)
    return output, scores

## 1. Forward keeps an online-softmax state per query row

One program owns a query tile. It loads that Q tile once, streams every K/V tile, and carries three row-wise quantities: the running maximum `m`, exponential sum `ell`, and output numerator `acc`. The complete $L \times L$ score and probability matrices never reach HBM.

In [ ]:
# Complete Triton forward and launcher
@triton.jit
def _flash_fwd(
    Q, K, V, O, LSE,
    stride_b: tl.constexpr,
    stride_h: tl.constexpr,
    H: tl.constexpr,
    L: tl.constexpr,
    D: tl.constexpr,
    SCALE: tl.constexpr,
    CAUSAL: tl.constexpr,
    BLOCK: tl.constexpr,
):
    q_block = tl.program_id(0)
    bh = tl.program_id(1)
    b, h = bh // H, bh % H
    base = b * stride_b + h * stride_h

    rows = q_block * BLOCK + tl.arange(0, BLOCK)
    cols_in_tile = tl.arange(0, BLOCK)
    dims = tl.arange(0, D)
    q = tl.load(
        Q + base + rows[:, None] * D + dims[None, :],
        mask=rows[:, None] < L,
        other=0.0,
    )

    m = tl.full([BLOCK], -float("inf"), tl.float32)
    ell = tl.zeros([BLOCK], tl.float32)
    acc = tl.zeros([BLOCK, D], tl.float32)

    for start_n in range(0, L, BLOCK):
        cols = start_n + cols_in_tile
        k = tl.load(
            K + base + cols[:, None] * D + dims[None, :],
            mask=cols[:, None] < L,
            other=0.0,
        )
        v = tl.load(
            V + base + cols[:, None] * D + dims[None, :],
            mask=cols[:, None] < L,
            other=0.0,
        )

        scores = tl.dot(q, tl.trans(k)) * SCALE
        visible = (rows[:, None] < L) & (cols[None, :] < L)
        if CAUSAL:
            visible = visible & (rows[:, None] >= cols[None, :])
        scores = tl.where(visible, scores, -1.0e6)

        m_new = tl.maximum(m, tl.max(scores, axis=1))
        alpha = tl.exp(m - m_new)
        p = tl.exp(scores - m_new[:, None])
        p = tl.where(visible, p, 0.0)
        acc = acc * alpha[:, None] + tl.dot(p.to(q.dtype), v)
        ell = ell * alpha + tl.sum(p, axis=1)
        m = m_new

    out = acc / ell[:, None]
    tl.store(
        O + base + rows[:, None] * D + dims[None, :],
        out,
        mask=rows[:, None] < L,
    )
    tl.store(LSE + bh * L + rows, m + tl.log(ell), mask=rows < L)


def _check_inputs(q, k, v):
    if not (q.is_cuda and k.is_cuda and v.is_cuda):
        raise ValueError("q, k, and v must be CUDA tensors")
    if q.shape != k.shape or q.shape != v.shape:
        raise ValueError("q, k, and v must have the same shape")
    if not (q.is_contiguous() and k.is_contiguous() and v.is_contiguous()):
        raise ValueError("q, k, and v must be contiguous")
    if q.dtype not in (torch.float16, torch.bfloat16):
        raise ValueError("q, k, and v must use fp16 or bf16")
    if q.shape[-1] not in (64, 128):
        raise ValueError("this teaching kernel supports head dimension 64 or 128")


def flash_forward(q, k, v, causal=False):
    _check_inputs(q, k, v)
    B, H, L, D = q.shape
    o = torch.empty_like(q)
    lse = torch.empty((B, H, L), device=q.device, dtype=torch.float32)
    block = 64
    grid = (triton.cdiv(L, block), B * H)
    _flash_fwd[grid](
        q, k, v, o, lse,
        q.stride(0), q.stride(1),
        H=H, L=L, D=D, SCALE=D**-0.5, CAUSAL=causal, BLOCK=block,
        num_warps=4, num_stages=2,
    )
    return o, lse

In [ ]:
B, H, L, D = 1, 2, 128, 64
q = torch.randn((B, H, L, D), device=DEVICE, dtype=DTYPE) * 0.5
k = torch.randn_like(q) * 0.5
v = torch.randn_like(q) * 0.5

o_tri, lse_tri = flash_forward(q, k, v, causal=False)
o_ref, scores_ref = attention_reference(q, k, v, causal=False)
lse_ref = torch.logsumexp(scores_ref, dim=-1)

check_close("forward output", o_tri, o_ref)
check_close("saved LSE", lse_tri, lse_ref)

## 2. LSE is the compact bridge to backward

For one row, `P = exp(S - LSE)`. Saving one scalar per query row lets backward reconstruct any probability tile after recomputing its score tile. For comparison, at length 4096 the complete `S` and `P` contain thousands of times more values than LSE.

In [ ]:
example_L = 4096
full_s_and_p = 2 * B * H * example_L * example_L
saved_lse = B * H * example_L
print(f"Full S and P: {full_s_and_p:,} values")
print(f"Saved LSE:    {saved_lse:,} values")
print(f"Ratio:        {full_s_and_p / saved_lse:,.0f}x")

## 3. Backward reconstructs probabilities instead of storing them

The preprocess kernel forms $D_i=dO_i^\top O_i$. The main kernel then uses two ownership directions. A KV-owned traversal finishes one `dK` and `dV` tile without atomics; a query-owned traversal finishes one `dQ` tile. Both traversals recompute scores and recover probabilities from the saved LSE.

In [ ]:
# Complete Triton backward: preprocess, KV-owned traversal, and query-owned traversal
@triton.jit
def _flash_bwd_preprocess(
    O, DO, Delta,
    stride_b: tl.constexpr,
    stride_h: tl.constexpr,
    H: tl.constexpr,
    L: tl.constexpr,
    D: tl.constexpr,
    BLOCK: tl.constexpr,
):
    block = tl.program_id(0)
    bh = tl.program_id(1)
    b, h = bh // H, bh % H
    base = b * stride_b + h * stride_h
    rows = block * BLOCK + tl.arange(0, BLOCK)
    dims = tl.arange(0, D)

    o = tl.load(
        O + base + rows[:, None] * D + dims[None, :],
        mask=rows[:, None] < L,
        other=0.0,
    )
    do = tl.load(
        DO + base + rows[:, None] * D + dims[None, :],
        mask=rows[:, None] < L,
        other=0.0,
    ).to(tl.float32)
    delta = tl.sum(o * do, axis=1)
    tl.store(Delta + bh * L + rows, delta, mask=rows < L)


@triton.jit
def _flash_bwd(
    Q, K, V, DO, DQ, DK, DV, LSE, Delta,
    stride_b: tl.constexpr,
    stride_h: tl.constexpr,
    H: tl.constexpr,
    L: tl.constexpr,
    D: tl.constexpr,
    SCALE: tl.constexpr,
    CAUSAL: tl.constexpr,
    BLOCK: tl.constexpr,
):
    tile = tl.program_id(0)
    bh = tl.program_id(1)
    b, h = bh // H, bh % H
    base = b * stride_b + h * stride_h
    in_tile = tl.arange(0, BLOCK)
    dims = tl.arange(0, D)

    # Traversal 1: one program owns a KV tile and finishes dK and dV.
    cols = tile * BLOCK + in_tile
    k = tl.load(
        K + base + cols[:, None] * D + dims[None, :],
        mask=cols[:, None] < L,
        other=0.0,
    )
    v = tl.load(
        V + base + cols[:, None] * D + dims[None, :],
        mask=cols[:, None] < L,
        other=0.0,
    )
    dk = tl.zeros([BLOCK, D], tl.float32)
    dv = tl.zeros([BLOCK, D], tl.float32)

    for start_m in range(0, L, BLOCK):
        rows = start_m + in_tile
        q = tl.load(
            Q + base + rows[:, None] * D + dims[None, :],
            mask=rows[:, None] < L,
            other=0.0,
        )
        do = tl.load(
            DO + base + rows[:, None] * D + dims[None, :],
            mask=rows[:, None] < L,
            other=0.0,
        )
        lse = tl.load(LSE + bh * L + rows, mask=rows < L, other=0.0)
        delta = tl.load(Delta + bh * L + rows, mask=rows < L, other=0.0)

        scores = tl.dot(q, tl.trans(k)) * SCALE
        visible = (rows[:, None] < L) & (cols[None, :] < L)
        if CAUSAL:
            visible = visible & (rows[:, None] >= cols[None, :])
        scores = tl.where(visible, scores, -1.0e6)
        p = tl.exp(scores - lse[:, None])
        p = tl.where(visible, p, 0.0)
        dp = tl.dot(do, tl.trans(v)).to(tl.float32)
        ds = p * (dp - delta[:, None])

        dv += tl.dot(tl.trans(p.to(q.dtype)), do)
        dk += tl.dot(tl.trans(ds.to(q.dtype)), q) * SCALE

    tl.store(
        DK + base + cols[:, None] * D + dims[None, :],
        dk,
        mask=cols[:, None] < L,
    )
    tl.store(
        DV + base + cols[:, None] * D + dims[None, :],
        dv,
        mask=cols[:, None] < L,
    )

    # Traversal 2: the same program id owns a query tile and finishes dQ.
    rows = tile * BLOCK + in_tile
    q = tl.load(
        Q + base + rows[:, None] * D + dims[None, :],
        mask=rows[:, None] < L,
        other=0.0,
    )
    do = tl.load(
        DO + base + rows[:, None] * D + dims[None, :],
        mask=rows[:, None] < L,
        other=0.0,
    )
    lse = tl.load(LSE + bh * L + rows, mask=rows < L, other=0.0)
    delta = tl.load(Delta + bh * L + rows, mask=rows < L, other=0.0)
    dq = tl.zeros([BLOCK, D], tl.float32)

    for start_n in range(0, L, BLOCK):
        cols = start_n + in_tile
        k = tl.load(
            K + base + cols[:, None] * D + dims[None, :],
            mask=cols[:, None] < L,
            other=0.0,
        )
        v = tl.load(
            V + base + cols[:, None] * D + dims[None, :],
            mask=cols[:, None] < L,
            other=0.0,
        )

        scores = tl.dot(q, tl.trans(k)) * SCALE
        visible = (rows[:, None] < L) & (cols[None, :] < L)
        if CAUSAL:
            visible = visible & (rows[:, None] >= cols[None, :])
        scores = tl.where(visible, scores, -1.0e6)
        p = tl.exp(scores - lse[:, None])
        p = tl.where(visible, p, 0.0)
        dp = tl.dot(do, tl.trans(v)).to(tl.float32)
        ds = p * (dp - delta[:, None])
        dq += tl.dot(ds.to(q.dtype), k) * SCALE

    tl.store(
        DQ + base + rows[:, None] * D + dims[None, :],
        dq,
        mask=rows[:, None] < L,
    )

In [ ]:
# Launcher and PyTorch autograd wrapper
def flash_backward(q, k, v, o, lse, do, causal=False):
    _check_inputs(q, k, v)
    if o.shape != q.shape or do.shape != q.shape:
        raise ValueError("o and do must have the same shape as q")
    if not (o.is_contiguous() and do.is_contiguous()):
        raise ValueError("o and do must be contiguous")

    B, H, L, D = q.shape
    dq = torch.empty_like(q)
    dk = torch.empty_like(k)
    dv = torch.empty_like(v)
    delta = torch.empty((B, H, L), device=q.device, dtype=torch.float32)
    block = 64
    grid = (triton.cdiv(L, block), B * H)

    _flash_bwd_preprocess[grid](
        o, do, delta, q.stride(0), q.stride(1),
        H=H, L=L, D=D, BLOCK=block, num_warps=4,
    )
    _flash_bwd[grid](
        q, k, v, do, dq, dk, dv, lse, delta,
        q.stride(0), q.stride(1),
        H=H, L=L, D=D, SCALE=D**-0.5, CAUSAL=causal, BLOCK=block,
        num_warps=4, num_stages=1,
    )
    return dq, dk, dv


class _FlashAttention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, causal=False):
        o, lse = flash_forward(q, k, v, causal)
        ctx.save_for_backward(q, k, v, o, lse)
        ctx.causal = causal
        return o

    @staticmethod
    def backward(ctx, do):
        q, k, v, o, lse = ctx.saved_tensors
        dq, dk, dv = flash_backward(q, k, v, o, lse, do.contiguous(), ctx.causal)
        return dq, dk, dv, None


flash_attention = _FlashAttention.apply

In [ ]:
do = torch.randn_like(q)

q_tri, k_tri, v_tri = [x.detach().clone().requires_grad_(True) for x in (q, k, v)]
out_tri = flash_attention(q_tri, k_tri, v_tri, False)
grads_tri = torch.autograd.grad(out_tri, (q_tri, k_tri, v_tri), do)

q_ref, k_ref, v_ref = [x.detach().clone().requires_grad_(True) for x in (q, k, v)]
out_ref, _ = attention_reference(q_ref, k_ref, v_ref, causal=False)
grads_ref = torch.autograd.grad(out_ref, (q_ref, k_ref, v_ref), do)

for name, actual, expected in zip(("dQ", "dK", "dV"), grads_tri, grads_ref):
    check_close(name, actual, expected, atol=5e-2, rtol=5e-2)

## 4. Check dense and causal attention in fp16 and bf16

The final helper creates fresh inputs and repeats the forward, LSE, and backward comparisons. Running both mask modes matters because backward must reconstruct exactly the same attention function that forward evaluated.

In [ ]:
def check_case(*, causal, dtype):
    torch.manual_seed(0)
    shape = (1, 2, 128, 64)
    q0 = torch.randn(shape, device=DEVICE, dtype=dtype) * 0.5
    k0 = torch.randn(shape, device=DEVICE, dtype=dtype) * 0.5
    v0 = torch.randn(shape, device=DEVICE, dtype=dtype) * 0.5
    do = torch.randn_like(q0)

    q_ref, k_ref, v_ref = [x.detach().clone().requires_grad_(True) for x in (q0, k0, v0)]
    o_ref, scores_ref = attention_reference(q_ref, k_ref, v_ref, causal=causal)
    grads_ref = torch.autograd.grad(o_ref, (q_ref, k_ref, v_ref), do)

    q_tri, k_tri, v_tri = [x.detach().clone().requires_grad_(True) for x in (q0, k0, v0)]
    o_tri = flash_attention(q_tri, k_tri, v_tri, causal)
    grads_tri = torch.autograd.grad(o_tri, (q_tri, k_tri, v_tri), do)
    _, lse_tri = flash_forward(q0, k0, v0, causal)
    lse_ref = torch.logsumexp(scores_ref, dim=-1)

    torch.testing.assert_close(o_tri, o_ref, atol=3e-2, rtol=3e-2)
    torch.testing.assert_close(lse_tri, lse_ref, atol=3e-2, rtol=3e-2)
    for actual, expected in zip(grads_tri, grads_ref):
        torch.testing.assert_close(actual, expected, atol=5e-2, rtol=5e-2)


for causal in (False, True):
    check_case(causal=causal, dtype=torch.float16)
    print(f"fp16 causal={causal}: passed")

if torch.cuda.is_bf16_supported():
    for causal in (False, True):
        check_case(causal=causal, dtype=torch.bfloat16)
        print(f"bf16 causal={causal}: passed")
else:
    print("This GPU does not support bf16; bf16 checks skipped.")

print("All available checks passed.")

## Done

Forward and backward preserve the same dense attention function while changing which intermediates reach HBM. Forward saves `O` and one LSE scalar per row; backward spends extra matrix multiplications to reconstruct the probability tiles it needs.